### Ячейка 1. Импорты и настройки

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

### Ячейка 2. Загрузка данных

In [3]:
# Загружаем исходные train/test
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

Train shape: (751, 214)
Test shape: (250, 211)


,index,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,102.414420,95.757483,0.935000,5.466584,5.466584,0.719259,0.719259,0.681165,18.307692,...,1,0,0,0,0,0,0,0,0,0
1,1,0.044333,8.401080,189.500000,11.492712,11.492712,0.012350,-3.798024,0.769122,27.652174,...,0,1,0,0,0,0,0,0,0,0
2,2,4.437964,50.085589,11.285714,5.366084,5.366084,0.522930,0.522930,0.612606,24.608696,...,0,0,0,0,0,0,0,0,0,0
3,3,6.827881,682.788051,100.000000,13.317130,13.317130,0.020658,-4.829339,0.345823,12.400000,...,0,0,1,0,0,0,0,0,0,0
4,4,2.003253,70.001455,34.943894,6.320833,6.320833,0.300347,0.300347,0.562066,60.272727,...,0,0,0,0,0,0,0,0,0,0


### Ячейка 3. Разделение таргетов и признаков

In [5]:
# Имя столбца с индексом и список таргетов как в исходном CSV
id_col = "index"
targets = ["IC50, mM", "CC50, mM", "SI"]

# train_first: index + таргеты (будем хранить здесь только таргеты)
train_first = train[[id_col] + targets].copy()

# train_features_raw: только признаки (удаляем index и таргеты)
train_features_raw = train.drop(columns=targets).drop(columns=[id_col])

# Аналогично для test: отдельный index и блок признаков
test_first = test[[id_col]].copy()
test_features_raw = test.drop(columns=[id_col])

print("Train features shape:", train_features_raw.shape)
print("Test features shape:", test_features_raw.shape)

Train features shape: (751, 210)
Test features shape: (250, 210)


In [6]:
### Ячейка 4. Arctan‑преобразование признаков

In [7]:
# Применяем arctan ко всем числовым признакам
train_features_atan = np.arctan(train_features_raw)
test_features_atan = np.arctan(test_features_raw)

# Возвращаем в DataFrame с исходными именами колонок
train_features_atan = pd.DataFrame(
    train_features_atan,
    columns=train_features_raw.columns
)
test_features_atan = pd.DataFrame(
    test_features_atan,
    columns=test_features_raw.columns
)

# Быстрая проверка распределений
train_features_atan.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
MaxAbsEStateIndex,751.0,1.462860,0.057308,1.164130,1.459165,1.488995,1.495264,1.508118
MaxEStateIndex,751.0,1.462860,0.057308,1.164130,1.459165,1.488995,1.495264,1.508118
MinAbsEStateIndex,751.0,0.172225,0.151830,0.000039,0.048436,0.120781,0.283170,0.941867
MinEStateIndex,751.0,-0.425246,0.597480,-1.428755,-0.927474,-0.397190,0.072333,0.941867
qed,751.0,0.509877,0.170199,0.059496,0.416885,0.566810,0.638673,0.758323


### Ячейка 5. Удаление сильно коррелированных признакоd

In [8]:
# Считаем матрицу корреляций по модулю
corr_matrix = train_features_atan.corr().abs()

# Берём верхний треугольник матрицы, чтобы не дублировать пары
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Выбрасываем признаки, у которых есть корреляция > 0.9 хоть с кем-то
cols_to_drop = [
    col for col in upper.columns if any(upper[col] > 0.9)
]

print("Drop", len(cols_to_drop), "highly correlated features")

# Чистые признаки после корр-отбора
train_clean = train_features_atan.drop(columns=cols_to_drop)
test_clean = test_features_atan.drop(
    columns=[c for c in cols_to_drop if c in test_features_atan.columns]
)

print("Train clean shape:", train_clean.shape)
print("Test clean shape:", test_clean.shape)

Drop 38 highly correlated features
Train clean shape: (751, 172)
Test clean shape: (250, 172)


### Ячейка 6. Импутация пропусков и масштабирование

In [9]:
# Импьютер средним и стандартизация
imputer = SimpleImputer(strategy="mean")
scaler = StandardScaler()

X = train_clean.values
X_test = test_clean.values

# Заполняем пропуски средним
X_imputed = imputer.fit_transform(X)
X_test_imputed = imputer.transform(X_test)

# Стандартизуем признаки
X_scaled = scaler.fit_transform(X_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

### Ячейка 7. Подготовка финальных матриц признаков (без PCA)

In [10]:
# Вместо PCA используем все стандартизованные признаки как есть
X_pca = X_scaled
X_test_pca = X_test_scaled

# Дадим имена колонкам признаков
pca_cols = [f"F{i+1}" for i in range(X_pca.shape[1])]

# Собираем итоговый train_final: таргеты + признаки
train_final = pd.concat(
    [
        train_first.reset_index(drop=True),
        pd.DataFrame(X_pca, columns=pca_cols)
    ],
    axis=1
)

# Итоговый test_final: index + признаки
test_final = pd.concat(
    [
        test_first.reset_index(drop=True),
        pd.DataFrame(X_test_pca, columns=pca_cols)
    ],
    axis=1
)

print("Train final:", train_final.shape)
print("Test final:", test_final.shape)
train_final.head()

Train final: (751, 176)
Test final: (250, 173)


,index,"IC50, mM","CC50, mM",SI,F1,F2,F3,F4,F5,F6,...,F163,F164,F165,F166,F167,F168,F169,F170,F171,F172
0,0,102.414420,95.757483,0.935000,-1.274551,2.974459,1.756511,0.517953,-0.560988,-1.633143,...,4.115522,-0.116169,-0.103765,-0.036515,-0.036515,-0.240305,0.0,-0.272749,-0.203063,-0.081868
1,1,0.044333,8.401080,189.500000,0.369186,-1.053692,-1.487397,0.856926,0.279826,0.382072,...,-0.214951,8.608136,-0.103765,-0.036515,-0.036515,-0.240305,0.0,-0.272749,-0.203063,-0.081868
2,2,4.437964,50.085589,11.285714,-1.332403,2.040472,1.519169,0.233769,0.075969,0.039716,...,-0.214951,-0.116169,-0.103765,-0.036515,-0.036515,-0.240305,0.0,-0.272749,-0.203063,-0.081868
3,3,6.827881,682.788051,100.000000,0.575971,-0.998953,-1.576614,-1.040229,-1.743405,0.806441,...,-0.214951,-0.116169,9.637168,-0.036515,-0.036515,-0.240305,0.0,-0.272749,-0.203063,-0.081868
4,4,2.003253,70.001455,34.943894,-0.855086,0.787924,1.200876,0.012836,1.172596,-2.911582,...,-0.214951,-0.116169,-0.103765,-0.036515,-0.036515,-0.240305,0.0,-0.272749,-0.203063,-0.081868


### Ячейка 8. Обучение LightGBM с KFold (линейные таргеты)

In [11]:
features = pca_cols  # список признаков
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# OOF-предсказания и предсказания на тесте для каждого таргета
oof = {t: np.zeros(len(train_final)) for t in targets}
test_preds = {t: np.zeros(len(test_final)) for t in targets}

for t in targets:
    print("Target:", t)
    # Обучаемся прямо на исходных таргетах (без лог-преобразования)
    y = train_final[t].values

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_final)):
        print(f"  Fold {fold+1}")
        X_tr = train_final.iloc[tr_idx][features]
        X_val = train_final.iloc[val_idx][features]
        y_tr, y_val = y[tr_idx], y[val_idx]

        train_data = lgb.Dataset(X_tr, label=y_tr)
        valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

        params = {
            "objective": "regression",
            "metric": "rmse",
            "learning_rate": 0.03,
            "num_leaves": 15,
            "min_data_in_leaf": 20,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 1,
            "verbose": -1,
        }

        model = lgb.train(
            params=params,
            train_set=train_data,
            num_boost_round=5000,
            valid_sets=[train_data, valid_data],
            valid_names=["train", "valid"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=200),
                lgb.log_evaluation(period=100),
            ],
        )

        # Предсказания на валидации и на тесте
        preds_val = model.predict(
            X_val, num_iteration=model.best_iteration
        )
        preds_test = model.predict(
            test_final[features], num_iteration=model.best_iteration
        )

        oof[t][val_idx] = preds_val
        test_preds[t] += preds_test / kf.n_splits

    # RMSE считаем в исходной шкале таргета (линейные IC50, CC50, SI)
    mse_t = mean_squared_error(train_final[t].values, oof[t])
    rmse_t = np.sqrt(mse_t)
    print(f"{t} CV RMSE: {rmse_t:.4f}")

Target: IC50, mM
  Fold 1
Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 216.116	valid's rmse: 385.664
[200]	train's rmse: 172.522	valid's rmse: 388.317
[300]	train's rmse: 151.24	valid's rmse: 391.046
Early stopping, best iteration is:
[129]	train's rmse: 199.32	valid's rmse: 384.605
  Fold 2
Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 248.602	valid's rmse: 238.628
[200]	train's rmse: 203.928	valid's rmse: 254.281
Early stopping, best iteration is:
[66]	train's rmse: 274.939	valid's rmse: 234.485
  Fold 3
Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 239.879	valid's rmse: 335.835
[200]	train's rmse: 198.32	valid's rmse: 336.406
[300]	train's rmse: 174.181	valid's rmse: 335.945
[400]	train's rmse: 159.587	valid's rmse: 336.296
Early stopping, best iteration is:
[265]	train's rmse: 181.324	valid's rmse: 334.965
  Fold 4
Training until validation scores don't improve for 200 round

### Ячейка 9. Формирование сабмита

In [12]:
# Здесь никаких лог-преобразований, берём предсказания как есть
submission = pd.DataFrame({
    "index": test_final[id_col],
    "IC50":  test_preds["IC50, mM"],
    "CC50":  test_preds["CC50, mM"],
    "SI":    test_preds["SI"],
})

submission.to_csv("submission_lgbm_atan_linear.csv", index=False)
submission.head()

,index,IC50,CC50,SI
0,0,250.908622,397.287510,68.466455
1,1,221.654287,404.654334,68.791608
2,2,58.898043,317.449147,62.216604
3,3,308.169815,410.206254,61.195295
4,4,179.623736,320.324830,56.014272


### submission_lgbm_atan_linear.csv Mikhail Komarov · 13s ago 301.23197